In [1]:
import operator
from typing import TypedDict, Annotated, List
from langgraph.graph import StateGraph, END
from langchain.chat_models import init_chat_model
from pydantic import BaseModel, Field

c:\Users\kwonm\anaconda3\envs\nlp_env\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

In [3]:
# 1. 게임 상태(State) 정의
class TRPGState(TypedDict):
    job: str               # 캐릭터 직업
    progression: int       # 스토리 진행도 (0~100)
    stats: dict            # 캐릭터 스탯 (힘과 기술)
    # 누적되어야 하는 값들은 Annotated와 operator.add 사용
    inventory: Annotated[List[str], operator.add] 
    current_story: Annotated[List[str], operator.add] 
    player_input: str      # 플레이어의 롤플레잉 지시문
    is_valid_action: bool  # 행동 개연성 통과 여부
    refusal_reason: str    # (불통과 시) 반려 사유

# LLM  불러오기
llm = init_chat_model('gpt-4.1-mini')

# --- 응답 구조체 (Pydantic) 정의 ---
class ValidatorOutput(BaseModel):
    is_valid_action: bool = Field(description="행동이 물리적/논리적으로 가능한지 여부")
    refusal_reason: str = Field(description="거절 시 사유 (통과 시 빈 문자열)")

class StateManagerOutput(BaseModel):
    stats: dict = Field(description="업데이트된 스탯 {'strength': 숫자, 'skill': 숫자}")
    inventory_updates: List[str] = Field(description="새로 획득한 아이템 목록 (없으면 빈 리스트)")


In [4]:

# --- 2. 에이전트 노드 정의 ---

def validator_node(state: TRPGState):
    """행동 개연성 판단 에이전트"""
    # LLM이 무조건 ValidatorOutput 구조로 답변하도록 강제
    validator_llm = llm.with_structured_output(ValidatorOutput)
    
    prompt = f"""당신은 TRPG 개연성 판정관입니다.
    - 캐릭터 스탯: {state['stats']}
    - 플레이어 행동: {state['player_input']}
    - 현재 상황: {state['current_story'][-1] if state['current_story'] else '게임 시작 전'}
    
    플레이어의 행동이 현재 스탯과 상황에 비추어 가능한지 판정하세요. 
    스탯에 비해 너무 무리한 행동이거나 세계관에 맞지 않으면 거절하세요."""
    
    result = validator_llm.invoke(prompt)
    return {
        "is_valid_action": result.is_valid_action, 
        "refusal_reason": result.refusal_reason
    }

def state_manager_node(state: TRPGState):
    """상태 및 기록 에이전트 (10턴 제한)"""
    manager_llm = llm.with_structured_output(StateManagerOutput)
    
    prompt = f"""당신은 TRPG 시스템 기록관입니다.
    - 현재 스탯: {state['stats']}
    - 플레이어 행동: {state['player_input']}
    
    행동의 결과에 따라 스탯(strength, skill)을 소폭 조절하고, 아이템을 얻었다면 목록을 작성하세요."""
    
    result = manager_llm.invoke(prompt)
    
    # 전체 10턴으로 끝내기 위해 무조건 진행도 10 증가
    new_progression = state['progression'] + 10 
    
    return {
        "progression": new_progression,
        "stats": result.stats,
        "inventory": result.inventory_updates
    }

def storyteller_node(state: TRPGState):
    """스토리 생성 에이전트"""
    prog = state['progression']
    
    # 진행도(10턴)에 따른 스토리 분기
    if prog <= 10:
        instruction = "게임의 시작입니다. 사용자의 입력을 바탕으로 세계관과 배경을 설정하세요."
    elif prog <= 40:
        instruction = "모험의 초반부입니다. 서브퀘스트나 갈등의 단초를 던져주세요."
    elif prog <= 70:
        instruction = "스토리의 중반부입니다. 결말을 향해 달려가는 긴박한 전개를 작성하세요."
    elif prog <= 90:
        instruction = "클라이막스입니다. 최종 시련을 묘사하세요."
    else: # prog == 100
        instruction = "모험의 결말입니다. 플레이어의 선택을 되짚으며 대단원의 막을 내리세요."

    choice_instruction = "그리고 플레이어가 선택을 할 수 있게 2~3개의 선택지를 제시하세요." if prog < 100 else ""

    prompt = f"""당신은 TRPG 마스터입니다.
    - 진행도: {prog}/100
    - 직업: {state['job']}
    - 플레이어 행동: {state['player_input']}
    
    지시사항: {instruction} {choice_instruction}"""
    
    response = llm.invoke(prompt)
    
    # current_story가 리스트이므로 리스트 형태로 반환하여 누적시킴
    return {"current_story": [response.content]}




In [5]:
# --- 3. 라우팅 및 그래프 연결 ---
def check_validity(state: TRPGState):
    if state["is_valid_action"]:
        return "valid"
    return "invalid"

workflow = StateGraph(TRPGState)

workflow.add_node("validator", validator_node)
workflow.add_node("state_manager", state_manager_node)
workflow.add_node("storyteller", storyteller_node)

workflow.set_entry_point("validator")

workflow.add_conditional_edges(
    "validator",
    check_validity,
    {
        "valid": "state_manager",
        "invalid": END # 반려되면 상태 업데이트 없이 이번 턴 종료 (사용자 재입력 필요)
    }
)

workflow.add_edge("state_manager", "storyteller")
workflow.add_edge("storyteller", END)

trpg_app = workflow.compile()

In [6]:
def run_console_trpg():
    print("=== 🗡️ 10턴의 TRPG 모험을 시작합니다 ===")
    
    # 초기 상태 설정
    current_state = {
        "job": "전사",  # 임시로 전사로 시작
        "progression": 0, 
        "stats": {"strength": 10, "skill": 5}, 
        "inventory": [], 
        "current_story": ["당신은 눈을 떴습니다. 모험을 시작하려면 행동을 입력하세요."], 
        "player_input": "", 
        "is_valid_action": True, 
        "refusal_reason": ""
    }

    print(current_state["current_story"][0])

    # 진행도가 100이 될 때까지 10턴 반복
    while current_state["progression"] < 100:
        print("-" * 50)
        action = input("\n당신의 행동은? (종료하려면 'q' 입력): ")
        
        if action.lower() == 'q':
            print("모험을 포기했습니다.")
            break
            
        current_state["player_input"] = action
        
        # LangGraph 실행 (그래프가 반환한 새로운 상태로 덮어쓰기)
        current_state = trpg_app.invoke(current_state)
        
        # 개연성 검증 실패 시
        if not current_state["is_valid_action"]:
            print(f"\n[행동 불가] ❌ {current_state['refusal_reason']}")
            print("다른 행동을 선택해 주세요.")
            continue # 진행도는 오르지 않고 다시 입력받음
            
        # 성공적으로 스토리가 진행되었을 때 결과 출력
        print("\n[게임 마스터] 🎲")
        # current_story 리스트의 가장 마지막(최신) 스토리를 출력
        print(current_state["current_story"][-1])
        
        # 현재 스탯 및 상황 요약 출력
        print(f"\n📈 진행도: {current_state['progression']}/100")
        print(f"💪 스탯: 힘 {current_state['stats']['strength']} / 기술 {current_state['stats']['skill']}")
        if current_state["inventory"]:
            print(f"🎒 인벤토리: {', '.join(current_state['inventory'])}")

    if current_state["progression"] >= 100:
        print("\n=== 🎉 모험이 끝났습니다! ===")

# 콘솔 실행
# run_console_trpg()